# CDK4–palbociclib affinity with optional MSA

Predict a protein–ligand structure, read pIC50 and binding probability, and inspect the three affinity embeddings.

Tested with **Boltz2 NIM 1.9.0 / client 0.6.0**.
From the repository root, install `python -m pip install -e ".[notebooks]"`
and start `jupyter lab examples/notebooks` in that environment.
Use a [ready NIM server](../../docs/deployment.md); `BOLTZ2_NIM_URL`
defaults to `http://localhost:8000`. Run cells in order.

[Setup and output files](../README.md#notebooks) ·
[NIM release notes](https://docs.nvidia.com/nim/bionemo/boltz2/1.9.0/release-notes.html)

Copyright (c) 2025-2026 NVIDIA Corporation.

In [ ]:
import os
from pathlib import Path
import boltz2_client
from boltz2_client import Boltz2Client, Polymer, Ligand, PredictionRequest, AlignmentFileRecord

client = Boltz2Client(
    base_url=os.getenv("BOLTZ2_NIM_URL", "http://localhost:8000"), timeout=1200,
)
metadata = await client.get_service_metadata()
print(f"Client {boltz2_client.__version__}; connected NIM {metadata.version} (tested: 1.9.0)")
output_dir = Path("notebook_outputs") / "cdk4"
output_dir.mkdir(parents=True, exist_ok=True)

## Define the protein and ligand

Inputs: full-length human CDK4 ([UniProt P11802](https://www.uniprot.org/uniprotkb/P11802/entry))
and palbociclib ([PubChem CID 5330286](https://pubchem.ncbi.nlm.nih.gov/compound/5330286)).
This request contains CDK4 and the ligand, without a cyclin partner.

In [ ]:
CDK4_SEQUENCE = 'MATSRYEPVAEIGVGAYGTVYKARDPHSGHFVALKSVRVPNGGGGGGGLPISTVREVALLRRLEAFEHPNVVRLMDVCATSRTDREIKVTLVFEHVDQDLRTYLDKAPPPGLPAETIKDLMRQFLRGLDFLHANCIVHRDLKPENILVTSGGTVKLADFGLARIYSYQMALTPVVVTLWYRAPEVLLQSTYATPVDMWSVGCIFAEMFRRKPLFCGNSEADQLGKIFDLIGLPPEDDWPRDVSLPRGAFPPRGPRPVQSVVPEMEESGAQLLLEMLTFNPHKRISAFRALQHSYLHKDEGNPE'
PALBOCICLIB_SMILES = 'CC1=C(C(=O)N(C2=NC(=NC=C12)NC3=NC=C(C=C3)N4CCNCC4)C5CCCC5)C(=O)C'

# Sources: UniProt P11802; PubChem CID 5330286.

## Optional: supply an alignment

Set `CDK4_MSA_PATH` to an existing A3M file, or set `MSA_NIM_URL` to a separate
MSA Search NIM 2.3.0 service. File input takes precedence. `MSA_DATABASE` selects
an installed database (default: `pdb70_220313`). The search uses that service's
existing databases; it does not download databases.

Without either setting, the notebook predicts without MSA information. Failed
searches and mismatched query sequences stop execution.

In [ ]:
from boltz2_client import A3MParser, MSAFormatConverter

msa = None
alignment = None
msa_path = os.getenv("CDK4_MSA_PATH")
msa_url = os.getenv("MSA_NIM_URL")
if msa_path:
    alignment = Path(msa_path).read_text()
elif msa_url:
    client.configure_msa_search(msa_endpoint_url=msa_url)
    database = os.getenv("MSA_DATABASE", "pdb70_220313")
    search_result = await client.search_msa(sequence=CDK4_SEQUENCE, databases=[database])
    alignment = MSAFormatConverter.extract_alignment(search_result, "a3m")
    if not alignment:
        raise ValueError("MSA Search returned no A3M alignment")
    print("MSA database:", database)
else:
    print("No MSA supplied: predicting without evolutionary alignment information.")

if alignment is not None:
    if A3MParser.parse(alignment).query_sequence != CDK4_SEQUENCE:
        raise ValueError("A3M query sequence must match CDK4_SEQUENCE")
    msa = {"custom": {"a3m": AlignmentFileRecord(alignment=alignment, format="a3m", rank=0)}}
    (output_dir / "input.a3m").write_text(alignment)

## Predict affinity and embeddings

Enable affinity and embeddings on the same ligand. One affinity diffusion
sample keeps this demonstration short; select sampling for your evaluation
protocol. pIC50 and binding probability are distinct model outputs, and the
converted IC50-like values below are predictions rather than measurements.

In [ ]:
request = PredictionRequest(
    polymers=[Polymer(id="A", molecule_type="protein", sequence=CDK4_SEQUENCE, msa=msa)],
    ligands=[Ligand(id="L1", smiles=PALBOCICLIB_SMILES, predict_affinity=True,
                    output_affinity_embedding=True)],
    diffusion_samples_affinity=1,
)
result = await client.predict(request, output_dir=output_dir)
affinity = result.affinities["L1"]
print("Predicted pIC50:", affinity.affinity_pic50)
print("Binding probability:", affinity.affinity_probability_binary)
print("Predicted IC50-like values (nM):", [10 ** (9 - value) for value in affinity.affinity_pic50])
for field in ("affinity_embedding", "model_1_affinity_embedding", "model_2_affinity_embedding"):
    embedding = getattr(affinity, field)
    print(field, "shape:", (len(embedding), len(embedding[0])))
print("Confidence:", result.confidence_scores)
print("Saved structures, scores and embeddings:", output_dir.resolve())

In [ ]:
import py3Dmol

view = py3Dmol.view(width=800, height=450)
view.addModel(result.structures[0].structure, "cif")
view.setStyle({"cartoon": {"color": "lightblue"}})
view.addStyle({"hetflag": True}, {"stick": {}})
view.zoomTo()
view.show()